<a href="https://colab.research.google.com/github/cruzwithme/U2/blob/main/1_AI_Research_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
!pip install gradio

In [26]:
  !pip install pymupdf


In [27]:
!pip install openai

In [28]:
!pip install requests

In [29]:
!pip install beautifulsoup4

In [30]:
from io import BytesIO
import requests  # ✅ Add this
from bs4 import BeautifulSoup  # ✅ And this
import fitz  # ✅ Also required for PyMuPDF
import gradio as gr
from openai import OpenAI

In [31]:
# Function to analyze the text with GPT-4

def analyze_text(api_key, input_type, pdf_file, url):
  try:
    client= OpenAI(api_key=api_key)

    if input_type == "PDF" and pdf_file is not None:
      text = extract_text_from_pdf(pdf_file.name)
    elif input_type == "URL" and url:
      text = extract_text_from_url(url)
    else:
      return "Please provide valid input"

    prompt = f"""
    Analyze the following content and return:
    -Bullet point summary of key ideas
    -Any contradictions found
    -Tags for major topics or themes

    Content:
    {text[:6000]}
    """

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "You are an expert research assistant"},
            {"role": "user", "content": prompt}
        ]

    )

    result = response.choices[0].message.content
    return result
  except Exception as e:
    return f"Error{e}"

In [32]:
from io import BytesIO
# Function to extract text from PDF file

def extract_text_from_pdf(file):
  with open(file, 'rb') as f:
    file_data = f.read()
  doc = fitz.open(stream=BytesIO(file_data), filetype="pdf")
  return "\n".join([page.get_text() for page in doc])

# Function to extract text from URL

def extract_text_from_url(url):
  response = requests.get(url)
  soup = BeautifulSoup(response.text, 'html.parser')
  paragraphs = soup.find_all('p')
  return "\n".join(p.get_text()for p in paragraphs)


In [33]:
import gradio as gr
from openai import OpenAI

# Gradio UI
def create_ui():
  with gr.Blocks() as demo:
    api_key = gr.Textbox(label="OpenAI API Key",type="password")
    input_type = gr.Radio(["PDF", "URL"], label="Input Type")
    pdf_file = gr.File(label="upload PDF (optional)", file_types=[".pdf"])
    url = gr.Textbox(label="URL (optional)")
    output = gr.Textbox(label = "Output Summary")

    def wrapped_analyze(api_key_val, input_type_val, pdf_val, url_val):
      return analyze_text(api_key_val, input_type_val, pdf_val, url_val)

    run_btn = gr.Button("Run Analysis")
    run_btn.click(wrapped_analyze, inputs=[api_key, input_type, pdf_file, url], outputs=output)

  demo.launch(share=True)
create_ui()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://51da9728701a317c67.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
